# MOSAIC P0/P1 Colab Runner

This notebook mounts the dedicated Drive workspace, pulls the committed MOSAIC code, runs the P0 contract gate, performs a short random smoke, and optionally runs the P1 PPO profile.

**Do not enable PPO or recurrent PPO until the P0 gate passes. This notebook does not enable MOSAIC memory, skills, or planning.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/sahiltolani30/learn-a-world.git'
REPO_DIR = Path('/content/learn-a-world')
DRIVE_ROOT = Path('/content/drive/MyDrive/learn-a-world/mosaic')
print('Drive root:', DRIVE_ROOT)

In [ ]:
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'mosaic/requirements.txt'], check=True)
print('Dependencies installed')

In [ ]:
for name in ('checkpoints', 'logs', 'shards', 'results', 'manifests', 'caches', 'notebooks'):
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
print('MOSAIC Drive folders ready:', DRIVE_ROOT)

## P0 contract gate

This is the required first run. It checks the exact action signature, pixel-only boundary, opaque permutation, empty metadata path, checkpoint roundtrip, and source integrity.

In [ ]:
p0_run = 'p0_contract_colab'
subprocess.run([
    sys.executable, '-m', 'mosaic.colab.run_phase',
    '--phase', 'p0_contract',
    '--drive-root', str(DRIVE_ROOT),
    '--run-id', p0_run,
], check=True)
print('P0 PASS: continue only if the command completed successfully.')

## P1 random smoke

This verifies end-to-end collection and Drive artifact writing without claiming competence.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'mosaic.colab.run_phase',
    '--phase', 'p1_baseline', '--algorithm', 'random',
    '--eval-episodes', '2', '--max-episode-steps', '1000',
    '--drive-root', str(DRIVE_ROOT), '--run-id', 'p1_random_smoke',
], check=True)
print('Random smoke complete')

## P1 PPO profile (manual unlock)

Leave `RUN_PPO_PROFILE = False` until P0 passes and you are ready to collect the first short profile. This is not the full training run.

In [ ]:
RUN_PPO_PROFILE = False
if RUN_PPO_PROFILE:
    subprocess.run([
        sys.executable, '-m', 'mosaic.colab.run_phase',
        '--phase', 'p1_baseline', '--algorithm', 'ppo',
        '--train-steps', '10000', '--eval-episodes', '5',
        '--max-episode-steps', '1000', '--device', 'auto',
        '--drive-root', str(DRIVE_ROOT), '--run-id', 'p1_profile_001',
    ], check=True)
else:
    print('PPO profile is locked. Set RUN_PPO_PROFILE=True only after P0 PASS.')

In [ ]:
import json

for summary in sorted((DRIVE_ROOT / 'results').glob('*/summary.json')):
    payload = json.loads(summary.read_text())
    print(summary.parent.name, payload.get('metrics', {}))